#### CELDA 1: CONTROL DE ENTORNO E IMPORTACIÓN

In [38]:
import os
import datetime
import numpy as np
import pandas as pd

# Desactivar advertencias de asignación encadenada para un código de producción limpio
pd.options.mode.chained_assignment = None

# Configuración rígida de rutas relativas desde la carpeta notebooks/
RUTA_RAW = "../data/raw"
RUTA_PROCESSED = "../data/processed"

print("========================================================================")
print("             PIPELINE DE ETL INICIALIZADO - FISIMart S.A.C.             ")
print("========================================================================")
print(f"● Origen de Datos (Raw)      : {RUTA_RAW}/")
print(f"● Destino del Datamart      : {RUTA_PROCESSED}/")
print("========================================================================")

             PIPELINE DE ETL INICIALIZADO - FISIMart S.A.C.             
● Origen de Datos (Raw)      : ../data/raw/
● Destino del Datamart      : ../data/processed/


#### CELDA 2: EXTRACCIÓN Y DIAGNÓSTICO DE CALIDAD INICIAL (REPORTE ANTES DEL ETL)

In [40]:
import os
import datetime
import numpy as np
import pandas as pd

# Desactivar advertencias de asignación encadenada para un código de producción limpio
pd.options.mode.chained_assignment = None

# Configuración de rutas relativas desde la carpeta notebooks/
RUTA_RAW = "../data/raw"
RUTA_PROCESSED = "../data/processed"

print(f"● Origen de Datos (Raw)      : {RUTA_RAW}/")
print(f"● Destino del Datamart      : {RUTA_PROCESSED}/")
print("========================================================================")

print("Cargando set de datos crudos desde el Data Lake (Raw)...")

# 1. EXTRACCIÓN: Carga de fuentes transaccionales crudas
clientes_raw = pd.read_csv(f"{RUTA_RAW}/clientes_raw.csv", sep=";", encoding="utf-8")
productos_raw = pd.read_csv(f"{RUTA_RAW}/productos_raw.csv", sep=";", encoding="utf-8")
tiendas_raw = pd.read_csv(f"{RUTA_RAW}/tiendas_raw.csv", sep=";", encoding="utf-8")
tiempo_raw = pd.read_csv(f"{RUTA_RAW}/tiempo_raw.csv", sep=";", encoding="utf-8", keep_default_na=True)
promociones_raw = pd.read_csv(f"{RUTA_RAW}/promociones_raw.csv", sep=";", encoding="utf-8")
ventas_raw = pd.read_csv(f"{RUTA_RAW}/ventas_raw.csv", sep=";", encoding="utf-8")

# =========================================================================
# INSPECCIÓN Y REPORTE DE AUDITORÍA PROFUNDO (PRE-ETL)
# =========================================================================
print("\n" + "=" * 95)
print("             PERFILADO DE DATOS INICIAL (FISIMart S.A.C. - 2026)             ")
print("=" * 95)

# Mapeo exacto de las variables recién extraídas
mapeo_auditoria = {
    "Clientes Crudos (raw)": clientes_raw,
    "Productos Crudos (raw)": productos_raw,
    "Tiendas Crudas (raw)": tiendas_raw,
    "Tiempo Crudo (raw)": tiempo_raw,
    "Promociones Crudas (raw)": promociones_raw,
    "Ventas Crudas (Hechos raw)": ventas_raw
}

for titulo, dataframe in mapeo_auditoria.items():
    print(f"\n ENTIDAD: {titulo.upper()}")
    print(f" -> Volumen total de filas encontradas: {len(dataframe)}")
    print(f" -> Cantidad de registros duplicados idénticos: {dataframe.duplicated().sum()} filas")
    
    # 1. CONTAMINANTE: NULOS TOTALES DE LA TABLA
    filas_con_nulos = dataframe.isna().any(axis=1).sum()
    print(f" -> Cantidad de filas con valores nulos (NaN): {filas_con_nulos} filas")
    
    # 2. DIAGNÓSTICO DE OTROS CONTAMINANTES (Conteos absolutos)
    print(" -> Diagnóstico de Otras Anomalías detectadas:")
    
    # A. Fechas con formato corrupto
    columnas_fecha = [col for col in dataframe.columns if "fecha" in col]
    total_fechas_corruptas = 0
    for col in columnas_fecha:
        fechas_mal_formato = dataframe[~dataframe[col].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$') & dataframe[col].notna()][col]
        total_fechas_corruptas += len(fechas_mal_formato)
    
    if total_fechas_corruptas > 0:
        print(f"    [Formato] Filas con fechas fuera del estándar YYYY-MM-DD: {total_fechas_corruptas} registros")
    else:
        if len(columnas_fecha) > 0:
            print("  [Formato] 0 filas con problemas de formato de fecha.")

    # B. Texto Sucio / Variantes en Categorías
    if "categoria" in dataframe.columns:
        categorias_unicas_raw = dataframe["categoria"].dropna().nunique()
        categorias_unicas_limpias = dataframe["categoria"].dropna().astype(str).str.strip().str.lower().nunique()
        variantes_redundantes = categorias_unicas_raw - categorias_unicas_limpias
        if variantes_redundantes > 0:
            print(f"   [Tipografía] Variantes redundantes de texto creadas por errores de tipeo/espacios: {variantes_redundantes} variantes")
        else:
            print("    [Tipografía] 0 problemas de texto detectados en categorías.")

    # C. Outliers de Negocio (Precios cero y Cantidades excesivas)
    if "precio_unitario" in dataframe.columns:
        precios_cero = (dataframe["precio_unitario"] == 0).sum()
        if precios_cero > 0:
            print(f"    [Negocio] Filas con 'precio_unitario' igual a 0.00: {precios_cero} registros")
            
    if "cantidad" in dataframe.columns:
        cantidades_anormales = (dataframe["cantidad"] == 500).sum()
        if cantidades_anormales > 0:
            print(f"    [Inventario] Filas con 'cantidad' atípica (igual a 500): {cantidades_anormales} registros")

    print("-" * 60)

# 3. EVALUACIÓN DE INTEGRIDAD REFERENCIAL PRE-ETL
# Validamos las ventas iniciales contra el maestro de productos inicial recién cargado
set_productos_maestros = set(productos_raw["id_producto"])
lineas_huerfanas_conteo = ventas_raw[~ventas_raw["id_producto"].isin(set_productos_maestros)]

print("\n" + "=" * 95)
print(" EVALUACIÓN GLOBAL DE INTEGRIDAD REFERENCIAL")
if len(lineas_huerfanas_conteo) > 0:
    print(f" -> ALERTA CRÍTICA: Se encontraron {len(lineas_huerfanas_conteo)} registros huérfanos en 'Fact_Ventas'.")
    print(f"    (Filas que apuntan al producto inexistente 'PROD9999')")
else:
    print(" -> Integridad Referencial 100% sólida entre Hechos y Dimensiones.")
print("=" * 95)

● Origen de Datos (Raw)      : ../data/raw/
● Destino del Datamart      : ../data/processed/
Cargando set de datos crudos desde el Data Lake (Raw)...

             PERFILADO DE DATOS INICIAL (FISIMart S.A.C. - 2026)             

 ENTIDAD: CLIENTES CRUDOS (RAW)
 -> Volumen total de filas encontradas: 5000
 -> Cantidad de registros duplicados idénticos: 0 filas
 -> Cantidad de filas con valores nulos (NaN): 75 filas
 -> Diagnóstico de Otras Anomalías detectadas:
    [Formato] Filas con fechas fuera del estándar YYYY-MM-DD: 5000 registros
------------------------------------------------------------

 ENTIDAD: PRODUCTOS CRUDOS (RAW)
 -> Volumen total de filas encontradas: 600
 -> Cantidad de registros duplicados idénticos: 0 filas
 -> Cantidad de filas con valores nulos (NaN): 12 filas
 -> Diagnóstico de Otras Anomalías detectadas:
   [Tipografía] Variantes redundantes de texto creadas por errores de tipeo/espacios: 7 variantes
------------------------------------------------------------


#### CELDA 3: PIPELINE DE TRANSFORMACIÓN (ETL) - PARTE 1: SANEAMIENTO DE DIMENSIONES

In [41]:
# =========================================================================
# SANEAMIENTO DE DIMENSIONES
# =========================================================================
print("="*95)
print(" 🛠️  ETL: FASE DE TRANSFORMACIÓN - PARTE 1: SANEAMIENTO DE DIMENSIONES MAESTRAS ")
print("="*95)


# Reparación de formatos de fecha mixtos/corruptos

def reparar_fechas_mixtas(columna):
    """
    Normaliza fechas corruptas al formato estándar rígido: YYYY-MM-DD
    Elimina residuos de tiempo '00:00:00' para evitar fallas en la auditoría.
    """
    meses_es = {
        'ene': '01', 'feb': '02', 'mar': '03', 'abr': '04', 'may': '05', 'jun': '06',
        'jul': '07', 'ago': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dic': '12'
    }
    
    def limpiar_celda(val):
        if pd.isna(val):
            return val
        
        # Si por alguna razón ya viene con la hora de Pandas, la limpiamos primero sacando los 10 primeros caracteres
        val_str = str(val).split(' ')[0].strip()
        
        # Caso 1: YYYY/MesTexto/DD -> (Ej: 2024/Dic/15)
        for mes, num in meses_es.items():
            if mes in val_str.lower():
                partes = val_str.split('/')
                if len(partes) == 3:
                    return f"{partes[0]}-{num}-{partes[2].zfill(2)}"
        
        # Caso 2: DD/MM/YYYY -> (Ej: 28/01/2024)
        if '/' in val_str:
            partes = val_str.split('/')
            if len(partes) == 3 and len(partes[2]) == 4:
                return f"{partes[2]}-{partes[1].zfill(2)}-{partes[0].zfill(2)}"
                
        return val_str

    # Aplicamos la lógica y forzamos a que el resultado se mantenga como string de 10 caracteres (YYYY-MM-DD)
    return columna.apply(limpiar_celda).astype(str).str.slice(0, 10)


# -------------------------------------------------------------------------
# 1. SANEAMIENTO: DIMENSIÓN CLIENTES
# -------------------------------------------------------------------------
print("\n[1/5] Saneando 'Clientes Crudos'...")
df_cliente_clean = clientes_raw.copy()

# A. Corregir el 100% de las fechas de alta corruptas
df_cliente_clean["fecha_alta"] = reparar_fechas_mixtas(df_cliente_clean["fecha_alta"])

# B. Imputar ingresos mensuales nulos con la mediana del negocio (evita sesgos por outliers)
mediana_ingresos = df_cliente_clean["ingresos_mensuales_aprox"].median()
df_cliente_clean["ingresos_mensuales_aprox"].fillna(mediana_ingresos, inplace=True)

# C. Completar distritos faltantes bajo la categoría de control "No Especificado"
df_cliente_clean["distrito"].fillna("No Especificado", inplace=True)


# -------------------------------------------------------------------------
# 2. SANEAMIENTO: DIMENSIÓN PRODUCTOS
# -------------------------------------------------------------------------
print("[2/5] Saneando 'Productos Crudos'...")
df_producto_clean = productos_raw.copy()

# A. Corregir la tipografía en 'categoria': eliminar espacios latentes y homogeneizar en formato Título
df_producto_clean["categoria"] = df_producto_clean["categoria"].astype(str).str.strip().str.title()

# B. Resolver nulos en la marca asignando el valor por defecto del catálogo
df_producto_clean["marca"].fillna("Sin Marca", inplace=True)


# -------------------------------------------------------------------------
# 3. SANEAMIENTO: DIMENSIÓN TIENDAS
# -------------------------------------------------------------------------
print("[3/5] Saneando 'Tiendas Crudas'...")
df_tienda_clean = tiendas_raw.copy()

# A. Control preventivo de nulos en nombres de sucursales
df_tienda_clean["nombre"].fillna("FISIMart - Sucursal Genérica", inplace=True)


# -------------------------------------------------------------------------
# 4. SANEAMIENTO: DIMENSIÓN TIEMPO
# -------------------------------------------------------------------------
print("[4/5] Saneando 'Tiempo Crudo'...")
df_tiempo_clean = tiempo_raw.copy()

# A. Resolver los nulos en 'es_feriado'. Si está vacío, asumimos día laboral estándar (0)
if "es_feriado" in df_tiempo_clean.columns:
    df_tiempo_clean["es_feriado"].fillna(0, inplace=True)


# -------------------------------------------------------------------------
# 5. SANEAMIENTO: DIMENSIÓN PROMOCIONES
# -------------------------------------------------------------------------
print("[5/5] Saneando 'Promociones Crudas'...")
df_promocion_clean = promociones_raw.copy()

# A. Control de nulos en el tipo de promoción comercial
df_promocion_clean["tipo"].fillna("Sin Promoción", inplace=True)

print("\n" + "="*95)
print("  las dimensiones maestras han sido purificadas. ")
print("="*95)

 🛠️  ETL: FASE DE TRANSFORMACIÓN - PARTE 1: SANEAMIENTO DE DIMENSIONES MAESTRAS 

[1/5] Saneando 'Clientes Crudos'...
[2/5] Saneando 'Productos Crudos'...
[3/5] Saneando 'Tiendas Crudas'...
[4/5] Saneando 'Tiempo Crudo'...
[5/5] Saneando 'Promociones Crudas'...

  las dimensiones maestras han sido purificadas. 


C:\Users\Personal\AppData\Local\Temp\ipykernel_10212\2828284743.py:58: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cliente_clean["ingresos_mensuales_aprox"].fillna(mediana_ingresos, inplace=True)
C:\Users\Personal\AppData\Local\Temp\ipykernel_10212\2828284743.py:61: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are settin

#### CELDA 4: PIPELINE DE TRANSFORMACIÓN (ETL) - PARTE 2: PURIFICACIÓN DE LA TABLA DE HECHOS

In [42]:
# =========================================================================
# PURIFICACIÓN DE LA TABLA DE HECHOS
# =========================================================================
print("="*95)
print(" 🛠️  ETL: FASE DE TRANSFORMACIÓN - PARTE 2: PURIFICACIÓN DE LA TABLA DE HECHOS (VENTAS) ")
print("="*95)

print("\nIniciando limpieza dura sobre la matriz de hechos Ventas...")
df_ventas_clean = ventas_raw.copy()

# -------------------------------------------------------------------------
# 1. ESTANDARIZACIÓN DE FECHAS EN TRANSACCIONES
# -------------------------------------------------------------------------
print(" -> [1/5] Corrigiendo formatos de fechas mixtas en transacciones...")
df_ventas_clean["fecha"] = reparar_fechas_mixtas(df_ventas_clean["fecha"])

# -------------------------------------------------------------------------
# 2. TRATAMIENTO DE VALORES NULOS EN DESCUENTOS
# -------------------------------------------------------------------------
print(" -> [2/5] Imputando valores nulos en descuentos comerciales...")
# Si la columna descuento es nula, significa que la venta no tuvo promoción (descuento = 0.0)
df_ventas_clean["descuento"].fillna(0.0, inplace=True)

# -------------------------------------------------------------------------
# 3. ELIMINACIÓN DE REDUNDANCIA (DUPLICADOS EXACTOS)
# -------------------------------------------------------------------------
filas_antes_duplicados = len(df_ventas_clean)
df_ventas_clean.drop_duplicates(inplace=True)
filas_eliminadas_duplicados = filas_antes_duplicados - len(df_ventas_clean)
print(f" -> [3/5] Eliminación de redundancia completada ({filas_eliminadas_duplicados} filas duplicadas borradas).")

# -------------------------------------------------------------------------
# 4. FILTRADO DE OUTLIERS OPERATIVOS DE NEGOCIO
# -------------------------------------------------------------------------
print(" -> [4/5] Removiendo outliers de negocio (cantidades absurdas y precios rotos)...")
# Filtramos de forma estricta: cantidades menores a 500 y precios mayores a cero
filas_antes_outliers = len(df_ventas_clean)
df_ventas_clean = df_ventas_clean[
    (df_ventas_clean["cantidad"] < 500) & 
    (df_ventas_clean["precio_unitario"] > 0.0)
]
filas_eliminadas_outliers = filas_antes_outliers - len(df_ventas_clean)
print(f"    • Registros anómalos de negocio eliminados: {filas_eliminadas_outliers} filas.")

# -------------------------------------------------------------------------
# 5. BLINDAJE DE INTEGRIDAD REFERENCIAL (REGISTROS HUÉRFANOS)
# -------------------------------------------------------------------------
print(" -> [5/5] Aplicando blindaje de integridad referencial contra el maestro de productos...")
# Usamos el set de IDs de la dimensión de productos limpia que procesamos en la celda anterior
set_productos_limpios = set(df_producto_clean["id_producto"])

filas_antes_integridad = len(df_ventas_clean)
df_ventas_clean = df_ventas_clean[df_ventas_clean["id_producto"].isin(set_productos_limpios)]
filas_eliminadas_huerfanas = filas_antes_integridad - len(df_ventas_clean)
print(f"    • Registros huérfanos (PROD9999) eliminados con éxito: {filas_eliminadas_huerfanas} filas.")

print("\n" + "="*95)
print(f"  ¡ETL PARTE 2 FINALIZADO! Tabla de hechos purificada.")
print(f"    Volumetría neta resultante: {len(df_ventas_clean)} filas útiles.")
print("="*95)

 🛠️  ETL: FASE DE TRANSFORMACIÓN - PARTE 2: PURIFICACIÓN DE LA TABLA DE HECHOS (VENTAS) 

Iniciando limpieza dura sobre la matriz de hechos Ventas...
 -> [1/5] Corrigiendo formatos de fechas mixtas en transacciones...
 -> [2/5] Imputando valores nulos en descuentos comerciales...
 -> [3/5] Eliminación de redundancia completada (245 filas duplicadas borradas).
 -> [4/5] Removiendo outliers de negocio (cantidades absurdas y precios rotos)...
    • Registros anómalos de negocio eliminados: 77 filas.
 -> [5/5] Aplicando blindaje de integridad referencial contra el maestro de productos...
    • Registros huérfanos (PROD9999) eliminados con éxito: 0 filas.

  ¡ETL PARTE 2 FINALIZADO! Tabla de hechos purificada.
    Volumetría neta resultante: 25421 filas útiles.


C:\Users\Personal\AppData\Local\Temp\ipykernel_10212\51023104.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_ventas_clean["descuento"].fillna(0.0, inplace=True)


#### CELDA 5: INGENIERÍA DE CARACTERÍSTICAS (MÉTRICAS DERIVADAS)

In [43]:
# =========================================================================
# INGENIERÍA DE CARACTERÍSTICAS (MÉTRICAS DERIVADAS)
# =========================================================================
print("="*95)
print("  ETL: FASE DE INGENIERÍA DE CARACTERÍSTICAS - CÁLCULO DE MÉTRICAS FINANCIERAS ")
print("="*95)

print("\nCalculando variables financieras vectorizadas de alto rendimiento...")

# 1. Cálculo del Importe Neto aplicando el factor de descuento comercial
# Fórmula: Importe = Cantidad * Precio Unitario * (1 - Descuento)
print(" -> [1/4] Vectorizando cálculo del 'importe' neto por línea de venta...")
df_ventas_clean["importe"] = round(
    df_ventas_clean["cantidad"] * df_ventas_clean["precio_unitario"] * (1.0 - df_ventas_clean["descuento"]), 2
)

# 2. Enriquecimiento de Datos: Incorporar el costo unitario base desde la Dimensión Producto limpia
print(" -> [2/4] Cruzando matriz de hechos con catálogo maestro para extraer 'costo' base...")
df_ventas_clean = df_ventas_clean.merge(
    df_producto_clean[["id_producto", "costo"]], 
    on="id_producto", 
    how="left"
)

# 3. Cálculo del Costo Total de la Línea de Venta
# Fórmula: Costo Total = Cantidad * Costo Unitario
print(" -> [3/4] Calculando el 'costo_total' acumulado de la transacción...")
df_ventas_clean["costo_total"] = round(df_ventas_clean["cantidad"] * df_ventas_clean["costo"], 2)

# 4. Cálculo del Margen de Ganancia Neto de la transacción
# Fórmula: Margen = Importe Neto - Costo Total
print(" -> [4/4] Extrayendo el 'margen' de utilidad neta final...")
df_ventas_clean["margen"] = round(df_ventas_clean["importe"] - df_ventas_clean["costo_total"], 2)

print("\n" + "="*95)
print(" ¡INGENIERÍA DE CARACTERÍSTICAS COMPLETADA! ")
print(" Columnas vectorizadas e inyectadas: 'importe', 'costo_total' y 'margen'.")
print("="*95)

  ETL: FASE DE INGENIERÍA DE CARACTERÍSTICAS - CÁLCULO DE MÉTRICAS FINANCIERAS 

Calculando variables financieras vectorizadas de alto rendimiento...
 -> [1/4] Vectorizando cálculo del 'importe' neto por línea de venta...
 -> [2/4] Cruzando matriz de hechos con catálogo maestro para extraer 'costo' base...
 -> [3/4] Calculando el 'costo_total' acumulado de la transacción...
 -> [4/4] Extrayendo el 'margen' de utilidad neta final...

 ¡INGENIERÍA DE CARACTERÍSTICAS COMPLETADA! 
 Columnas vectorizadas e inyectadas: 'importe', 'costo_total' y 'margen'.


#### CELDA 6: CARGA Y PERSISTENCIA EN DATAMART PROCESADO

In [44]:
# =========================================================================
# CARGA Y PERSISTENCIA EN DATAMART PROCESADO
# =========================================================================
print("="*95)
print(" ETL: FASE DE CARGA - ESTRUCTURACIÓN Y PERSISTENCIA EN ESQUEMA ESTRELLA ")
print("="*95)

print("\nEstructurando esquemas estrella definitivos según arquitectura corporativa...")

# 1. Definición estricta de variables analíticas finales para las tablas modificadas
columnas_clientes = ["id_cliente", "nombre", "sexo", "fecha_nacimiento", "edad", "distrito", "fecha_alta", "segmento_programa", "ingresos_mensuales_aprox"]
columnas_productos = ["id_producto", "nombre", "categoria", "subcategoria", "marca", "precio_lista", "costo"]
columnas_hechos_ventas = ["id_venta", "fecha", "id_cliente", "id_producto", "id_tienda", "id_promocion", "cantidad", "precio_unitario", "descuento", "importe", "costo_total", "margen"]

# 2. Generación de las Dataframes finales del Datamart
Dim_Cliente = df_cliente_clean[columnas_clientes]
Dim_Producto = df_producto_clean[columnas_productos]
Fact_Ventas = df_ventas_clean[columnas_hechos_ventas]

# Para Tienda, Tiempo y Promoción arrastramos sus estados saneados completos
Dim_Tienda = df_tienda_clean
Dim_Tiempo = df_tiempo_clean
Dim_Promocion = df_promocion_clean

print(" -> [1/2] Estructuras validadas. Iniciando persistencia física en disco...")

# 3. Guardado en la carpeta limpia ../data/processed/ con codificación y separadores estándar
Dim_Cliente.to_csv(f"{RUTA_PROCESSED}/Dim_Cliente.csv", sep=";", index=False, encoding="utf-8")
Dim_Producto.to_csv(f"{RUTA_PROCESSED}/Dim_Producto.csv", sep=";", index=False, encoding="utf-8")
Dim_Tienda.to_csv(f"{RUTA_PROCESSED}/Dim_Tienda.csv", sep=";", index=False, encoding="utf-8")
Dim_Tiempo.to_csv(f"{RUTA_PROCESSED}/Dim_Tiempo.csv", sep=";", index=False, encoding="utf-8")
Dim_Promocion.to_csv(f"{RUTA_PROCESSED}/Dim_Promocion.csv", sep=";", index=False, encoding="utf-8")
Fact_Ventas.to_csv(f"{RUTA_PROCESSED}/Fact_Ventas.csv", sep=";", index=False, encoding="utf-8")

print(" -> [2/2] Archivos CSV creados y validados de forma segura.")
print("\n" + "="*95)
print(" ¡ETL COMPLETO Y EXITOSO! El Datamart ha sido cargado con éxito en processed/. ")

 ETL: FASE DE CARGA - ESTRUCTURACIÓN Y PERSISTENCIA EN ESQUEMA ESTRELLA 

Estructurando esquemas estrella definitivos según arquitectura corporativa...
 -> [1/2] Estructuras validadas. Iniciando persistencia física en disco...
 -> [2/2] Archivos CSV creados y validados de forma segura.

 ¡ETL COMPLETO Y EXITOSO! El Datamart ha sido cargado con éxito en processed/. 


#### CELDA 7: REPORTE DE CALIDAD FINAL DE AUDITORÍA (POST-ETL)

In [45]:
# =========================================================================
# REPORTE DE CALIDAD FINAL DE AUDITORÍA (POST-ETL)
# =========================================================================
print("="*95)
print("             REPORTE DE CALIDAD FINAL CERTIFICADO (POST-ETL) - FISIMart S.A.C.          ")
print("="*95)

# Mapeo de las dimensiones estructuradas y la tabla de hechos final
mapeo_procesado = {
    "Dim_Cliente (Limpia)": Dim_Cliente,
    "Dim_Producto (Limpia)": Dim_Producto,
    "Dim_Tienda (Limpia)": Dim_Tienda,
    "Dim_Tiempo (Limpia)": Dim_Tiempo,
    "Dim_Promocion (Limpia)": Dim_Promocion,
    "Fact_Ventas (Limpia)": Fact_Ventas
}

for titulo, df_final in mapeo_procesado.items():
    print(f"\n DATAMART ENTIDAD: {titulo.upper()}")
    print(f" -> Volumen útil resultante: {len(df_final)} filas")
    print(f" -> Cantidad de registros duplicados idénticos: {df_final.duplicated().sum()} filas")
    
    # 1. CONTROL DE NULOS ABSOLUTOS (Debe salir 0)
    filas_con_nulos_final = df_final.isna().any(axis=1).sum()
    print(f" -> Cantidad de filas con valores nulos (NaN): {filas_con_nulos_final} filas")
    
    # 2. CONTROL DE FORMATOS, TIPOGRAFÍA Y NEGOCIO
    print(" -> Diagnóstico de Control de Calidad:")
    
    # A. Fechas (Estandarizadas)
    columnas_fecha_final = [col for col in df_final.columns if "fecha" in col]
    total_fechas_corruptas_final = 0
    for col in columnas_fecha_final:
        fechas_mal_formato_final = df_final[~df_final[col].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$') & df_final[col].notna()][col]
        total_fechas_corruptas_final += len(fechas_mal_formato_final)
    print(f"    [Formato] Filas con fechas fuera del estándar YYYY-MM-DD: {total_fechas_corruptas_final} registros")

    # B. Tipografía en Categorías (Normalizadas)
    if "categoria" in df_final.columns:
        categorias_unicas_raw = df_final["categoria"].dropna().nunique()
        categorias_unicas_limpias = df_final["categoria"].dropna().astype(str).str.strip().str.lower().nunique()
        variantes_redundantes_final = columnas_invalidas = categorias_unicas_raw - categorias_unicas_limpias
        print(f"    [Tipografía] Variantes redundantes de texto en categorías: {variantes_redundantes_final} variantes")

    # C. Outliers Operativos (Removidos)
    if "precio_unitario" in df_final.columns:
        precios_cero_final = (df_final["precio_unitario"] == 0).sum()
        print(f"    [Negocio] Filas con 'precio_unitario' igual a 0.00: {precios_cero_final} registros")
            
    if "cantidad" in df_final.columns:
        cantidades_anormales_final = (df_final["cantidad"] == 500).sum()
        print(f"    [Inventario] Filas con 'cantidad' atípica (igual a 500): {cantidades_anormales_final} registros")

    print("-" * 60)

# 3. CERTIFICACIÓN DE INTEGRIDAD REFERENCIAL POST-ETL
set_prod_finales = set(Dim_Producto["id_producto"])
huerfanos_finales = len(Fact_Ventas[~Fact_Ventas["id_producto"].isin(set_prod_finales)])

print("\n" + "="*95)
print(" CERTIFICACIÓN GLOBAL DE INTEGRIDAD REFERENCIAL")
if huerfanos_finales > 0:
    print(f" -> ALERTA RESIDUAL: Aún persisten {huerfanos_finales} filas huérfanas en Fact_Ventas.")
else:
    print(f" -> Integridad Referencial 100% sólida entre Hechos y Dimensiones ({huerfanos_finales} huérfanos).")
print(f" -> ESTADO GENERAL DEL DATAMART: CERTIFICADO, LIMPIO Y LISTO PARA CONEXIÓN A POWER BI")
print("="*95)

             REPORTE DE CALIDAD FINAL CERTIFICADO (POST-ETL) - FISIMart S.A.C.          

 DATAMART ENTIDAD: DIM_CLIENTE (LIMPIA)
 -> Volumen útil resultante: 5000 filas
 -> Cantidad de registros duplicados idénticos: 0 filas
 -> Cantidad de filas con valores nulos (NaN): 0 filas
 -> Diagnóstico de Control de Calidad:
    [Formato] Filas con fechas fuera del estándar YYYY-MM-DD: 0 registros
------------------------------------------------------------

 DATAMART ENTIDAD: DIM_PRODUCTO (LIMPIA)
 -> Volumen útil resultante: 600 filas
 -> Cantidad de registros duplicados idénticos: 0 filas
 -> Cantidad de filas con valores nulos (NaN): 0 filas
 -> Diagnóstico de Control de Calidad:
    [Formato] Filas con fechas fuera del estándar YYYY-MM-DD: 0 registros
    [Tipografía] Variantes redundantes de texto en categorías: 0 variantes
------------------------------------------------------------

 DATAMART ENTIDAD: DIM_TIENDA (LIMPIA)
 -> Volumen útil resultante: 20 filas
 -> Cantidad de registros 